# Phase 2 – Build the Searchable ASR Corpus

## 0. Install the online retrieval dependencies

**What this does:** installs the E5/SentenceTransformers environment. Phase 2 does not need NeMo or Parakeet because transcription has already been completed by Phase 1.

In [1]:
import subprocess
import sys


packages = [
    "transformers==4.52.0",
    "sentence-transformers==5.4.0",
]

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    *packages,
]

print("Installing:")
print("\n".join(f"  {package}" for package in packages))

subprocess.run(command, check = True)

print("\nDependencies installed.")
print("Restart the Colab session once, then continue with Section 1.")

Installing:
  transformers==4.52.0
  sentence-transformers==5.4.0

Dependencies installed.
Restart the Colab session once, then continue with Section 1.


## 1. Setup and Production Tests

**What this verifies:** the production package exists on Drive and all deterministic tests still pass before we load a real neural model.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import importlib.metadata
import os
import sys


PROJECT_ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/"
    "aic26/asr_model_comparison"
)

PRODUCTION_ROOT = PROJECT_ROOT / "production"

assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert (PRODUCTION_ROOT / "asr_retrieval").exists()

os.chdir(PROJECT_ROOT)

if (str(PRODUCTION_ROOT) not in sys.path) :
    sys.path.insert(0, str(PRODUCTION_ROOT))


print("Project root:", PROJECT_ROOT)
print("Production package:", PRODUCTION_ROOT)

print("\nEnvironment")
print("Python:", sys.version.split()[0])
print("Transformers:", importlib.metadata.version("transformers"))
print("SentenceTransformers:", importlib.metadata.version("sentence-transformers"))


print("\nProduction files")
for path in sorted((PRODUCTION_ROOT / "asr_retrieval").glob("*.py")) :
    print(" ", path.name)


# Importing the package itself must remain lightweight.
assert "nemo" not in sys.modules
assert "sentence_transformers" not in sys.modules

import asr_retrieval

print("\nLightweight import")
print("NeMo imported:", "nemo" in sys.modules)
print("SentenceTransformers imported:", "sentence_transformers" in sys.modules)

assert "nemo" not in sys.modules
assert "sentence_transformers" not in sys.modules

Mounted at /content/drive
Project root: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Production package: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/production

Environment
Python: 3.12.13
Transformers: 4.52.0
SentenceTransformers: 5.4.0

Production files
  __init__.py
  artifacts.py
  audio.py
  bm25.py
  config.py
  dense.py
  postprocess.py
  schemas.py
  transcriber.py

Lightweight import
NeMo imported: False
SentenceTransformers imported: False


In [2]:
import subprocess
import sys


command = [
    sys.executable,
    "-m",
    "pytest",
    "tests/production/test_phase1.py",
    "tests/production/test_phase2.py",
    "-q",
]

print("Running production tests...\n")

subprocess.run(
    command,
    cwd = PROJECT_ROOT,
    check = True,
)

print("\nProduction tests PASS.")

Running production tests...


Production tests PASS.


## 2. Inspect the Phase 1 Transcripts We Are About to Index

Phase 1 produced durable transcript JSON files. Phase 2 will read these files and turn their searchable transcript windows into:

```
BM25 documents + E5 embeddings
```

In [3]:
import json


PHASE1_WORKSPACE = PROJECT_ROOT / "production_smoke" / "phase1"
TRANSCRIPTS_ROOT = PHASE1_WORKSPACE / "transcripts"

assert TRANSCRIPTS_ROOT.exists(), TRANSCRIPTS_ROOT

transcript_files = sorted(
    TRANSCRIPTS_ROOT.glob("*.json")
)

assert transcript_files, (
    f"No Phase 1 transcript JSON files found under {TRANSCRIPTS_ROOT}"
)


print("Phase 1 workspace:", PHASE1_WORKSPACE)
print("Transcript files:", len(transcript_files))

for path in transcript_files[:10] :
    print(" ", path.name)


sample = json.loads(
    transcript_files[0].read_text(
        encoding = "utf-8"
    )
)

windows = sample["windows"]
searchable_windows = [
    window
    for window in windows
    if window["eligible"]
]


print("\nExample video:", sample["video"]["video_id"])
print("All transcript windows:", len(windows))
print("Searchable transcript windows:", len(searchable_windows))
print("Not searchable:", len(windows) - len(searchable_windows))


print("\nFirst few transcript windows")

for window in windows[:5] :
    print("-" * 80)
    print("Window:", window["window_id"])
    print(
        "Time:",
        f'{window["start_s"]:.1f}s → {window["end_s"]:.1f}s',
    )
    print("Searchable:", window["eligible"])
    print("Text:", window["retrieval_text"][:400])

Phase 1 workspace: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/production_smoke/phase1
Transcript files: 1
  L21_V015.json

Example video: L21_V015
All transcript windows: 29
Searchable transcript windows: 29
Not searchable: 0

First few transcript windows
--------------------------------------------------------------------------------
Window: L21_V015_0000
Time: 0.0s → 60.0s
Searchable: True
Text: ch kính chào quý vị đây là chương trình sáu mươi giây sáng của đài truyền hình thành phố hồ chí minh và chương trình hôm nay sẽ có những nội dung đáng chú ý sau đây gió bỏ lệnh cấm xe ô tô cầu rạch miễu nhờ thử tải cầu kết thúc sớm hai ngày so với dự định nhà vệ sinh bệnh viện được nâng cấp đảm bảo sạch sẽ an toàn sa la la ốc đảo thu hút nhiều du khách đến ô men thưa quý vị lệnh cấm xe ô tô qua cầ
--------------------------------------------------------------------------------
Window: L21_V015_0001
Time: 45.0s → 105.0s
Searchable: True


## 3. Load and Test the Real E5 Model

**What this does:** loads our actual production E5 model and checks that it can encode text correctly on the T4.

E5 takes a piece of text and represents its meaning using 1024 floating-point numbers. Later, query embeddings are compared against transcript embeddings.

In [4]:
import numpy as np
import torch

from asr_retrieval.config import (
    ArtifactConfig,
    BM25Config,
    CorpusBuildConfig,
    E5Config,
    OfflineASRConfig,
)

from asr_retrieval.dense import E5Encoder


print("CUDA available:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "None",
)

assert torch.cuda.is_available()


config = CorpusBuildConfig(
    offline = OfflineASRConfig(),

    bm25 = BM25Config(),

    e5 = E5Config(
        device = "cuda",
        document_batch_size = 16,
    ),

    artifacts = ArtifactConfig(
        verify_file_hashes = True,
        load_embeddings_into_ram = True,
    ),
)


print("\nE5 configuration")
print("Model:", config.e5.model_name)
print("Revision:", config.e5.revision)
print("Embedding dimensions:", config.e5.dimension)
print("Document prefix:", repr(config.e5.document_prefix))


encoder = E5Encoder(
    config.e5,
)

print("\nLoading E5...")
encoder.load()

print("E5 loaded successfully.")
print("Model dimension:", encoder._model_dimension())


# Test one real query.
test_query = "một đàn hổ quý hiếm"

query_embedding = encoder.encode_query(
    test_query
)

print("\nEmbedding test")
print("Query:", test_query)
print("Shape:", query_embedding.shape)
print("dtype:", query_embedding.dtype)
print("L2 norm:", float(np.linalg.norm(query_embedding)))


assert query_embedding.shape == (1024,)
assert query_embedding.dtype == np.float32
assert np.isfinite(query_embedding).all()

assert np.isclose(
    np.linalg.norm(query_embedding),
    1.0,
    atol = 1e-5,
)

print("\nReal E5 encoding PASS.")

CUDA available: True
GPU: Tesla T4

E5 configuration
Model: intfloat/multilingual-e5-large-instruct
Revision: 274baa43b0e13e37fafa6428dbc7938e62e5c439
Embedding dimensions: 1024
Document prefix: ''

Loading E5...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

E5 loaded successfully.
Model dimension: 1024

Embedding test
Query: một đàn hổ quý hiếm
Shape: (1024,)
dtype: float32
L2 norm: 0.9999999403953552

Real E5 encoding PASS.


This proves:

```
text
↓
our pinned E5 model
↓
T4 GPU
↓
1024-dimensional float32 vector
↓
correct normalization
```

So we know the real dense-retrieval component works.

## 4. Build and Load One Real Searchable Corpus Package

We take `production_smoke/phase1/transcripts/` and build `production_smoke/phase2/releases/phase2-smoke-001/`.

The resulting folder is one self-contained searchable corpus package. Internally:

```
Phase 1 transcript JSON
        ↓
all transcript windows
        ↓
select searchable windows
        ↓
build BM25 from their text
        ↓
run E5 on their text
        ↓
save text + indexes + mappings together
```

In [5]:
from asr_retrieval.artifacts import (
    build_release,
    load_release,
    validate_release,
)

from asr_retrieval.dense import (
    exact_dense_scores,
)


PHASE2_ROOT = (
    PROJECT_ROOT
    / "production_smoke"
    / "phase2"
)

RELEASES_ROOT = (
    PHASE2_ROOT
    / "releases"
)

RELEASES_ROOT.mkdir(
    parents = True,
    exist_ok = True,
)


# Use a new ID if this one already exists.
release_001 = build_release(
    workspace = PHASE1_WORKSPACE,
    releases_root = RELEASES_ROOT,
    release_id = "phase2-smoke-001",
    config = config,
    encoder = encoder,
)


print("Searchable corpus package created:")
print(release_001)


# Validate every saved file and its recorded hash.
manifest = validate_release(
    release_001,
    verify_hashes = True,
)


# Load it exactly as the future search engine will.
loaded = load_release(
    release_001,
)


print("\nCorpus summary")
print("Videos:", loaded.manifest.video_count)
print(
    "All transcript windows:",
    loaded.manifest.physical_window_count,
)
print(
    "Searchable transcript windows:",
    loaded.manifest.eligible_window_count,
)

print(
    "E5 matrix:",
    loaded.embeddings.shape,
    loaded.embeddings.dtype,
)

print(
    "BM25 documents:",
    loaded.bm25.document_count,
)


assert loaded.embeddings.shape == (
    loaded.manifest.eligible_window_count,
    1024,
)

assert loaded.bm25.document_count == (
    loaded.manifest.eligible_window_count
)


# Every searchable E5 row must map back to the correct original window.
for searchable_index, physical_index in enumerate(
    loaded.eligible_to_physical
) :
    physical_index = int(physical_index)

    window = loaded.windows[
        physical_index
    ]

    assert window.eligible

    assert (
        loaded.eligible_window_ids[
            searchable_index
        ]
        == window.window_id
    )


# Check that all saved document vectors remain normalized.
norms = np.linalg.norm(
    loaded.embeddings,
    axis = 1,
)

assert np.allclose(
    norms,
    1.0,
    atol = 1e-5,
)


print("\nWindow-to-embedding mappings PASS.")
print("E5 matrix validation PASS.")
print("BM25 document count PASS.")


# Small functional test using the loaded database.
dense_scores = exact_dense_scores(
    query_embedding,
    loaded.embeddings,
)

bm25_scores = loaded.bm25.score(
    test_query
)


print("\nSearch component test")
print("Dense scores:", dense_scores.shape)
print("BM25 scores:", bm25_scores.shape)

assert dense_scores.shape == bm25_scores.shape
assert np.isfinite(dense_scores).all()
assert np.isfinite(bm25_scores).all()


# Show the top E5 transcript matches simply to inspect the mapping.
top_indices = np.argsort(
    -dense_scores
)[:5]

print("\nTop E5 transcript matches")

for rank, searchable_index in enumerate(
    top_indices,
    start = 1,
) :
    physical_index = int(
        loaded.eligible_to_physical[
            searchable_index
        ]
    )

    window = loaded.windows[
        physical_index
    ]

    print("-" * 80)
    print(
        f"{rank}.",
        window.video_id,
        window.window_id,
        f"score={dense_scores[searchable_index]:.4f}",
    )
    print(
        f"{window.start_s:.1f}s → {window.end_s:.1f}s"
    )
    print(window.retrieval_text[:500])

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Searchable corpus package created:
/content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/production_smoke/phase2/releases/phase2-smoke-001

Corpus summary
Videos: 1
All transcript windows: 29
Searchable transcript windows: 29
E5 matrix: (29, 1024) float32
BM25 documents: 29

Window-to-embedding mappings PASS.
E5 matrix validation PASS.
BM25 document count PASS.

Search component test
Dense scores: (29,)
BM25 scores: (29,)

Top E5 transcript matches
--------------------------------------------------------------------------------
1. L21_V015 L21_V015_0020 score=0.8799
900.0s → 960.0s
đây đâyốn là lo loại có nguy cơ tuyệt chủng nên phát hiện này làm dấy lên hy vọng về khả năng bảo tồn loài này theo tổ chức tái hoa xã chi lê các nhà bảo tồn đã bắt gặp và ghi lại hình ảnh một con hươu gê mun trên bờ biển ma ga la nét nhóm này khi đó đang tám hiểm khu vực sẽ trở thành công viên quốc gia kép ph rô gốt ở điểm cực nam của lục địa nam mỹ con hươu đượ

Suppose the output says:

* Videos: 1
* All transcript windows: 20
* Searchable transcript windows: 18
* E5 matrix: (18, 1024)
* BM25 documents: 18

That means:

```
1 video
↓
20 time windows exist
↓
18 have useful searchable transcripts
↓
BM25 indexes exactly those 18
↓
E5 stores exactly 18 × 1024 semantic vectors
```

## 5. Verify We Do Not Recompute Unchanged E5 Embeddings

The first package already contains the E5 vectors. If the transcripts have not changed, a new corpus package should simply reuse those vector instead of running E5 again.

In [6]:
release_002 = build_release(
    workspace = PHASE1_WORKSPACE,
    releases_root = RELEASES_ROOT,
    release_id = "phase2-smoke-002",
    config = config,

    # Deliberately no E5 encoder.
    encoder = None,

    # Reuse unchanged vectors from this package.
    previous_release = release_001,
)


loaded_002 = load_release(
    release_002,
)


print("Second searchable corpus package:")
print(release_002)


print("\nComparing corpus identity")
print(
    "Same corpus:",
    loaded.manifest.corpus_identity
    == loaded_002.manifest.corpus_identity,
)

print(
    "Same searchable-window mapping:",
    loaded.manifest.eligible_window_axis_sha256
    == loaded_002.manifest.eligible_window_axis_sha256,
)

print(
    "E5 vectors exactly identical:",
    np.array_equal(
        loaded.embeddings,
        loaded_002.embeddings,
    ),
)


assert (
    loaded.manifest.corpus_identity
    == loaded_002.manifest.corpus_identity
)

assert (
    loaded.manifest.eligible_window_axis_sha256
    == loaded_002.manifest.eligible_window_axis_sha256
)

assert np.array_equal(
    loaded.embeddings,
    loaded_002.embeddings,
)


print(
    "\nPASS: unchanged transcript windows reused "
    "their existing E5 embeddings."
)


encoder.close()

print("E5 model closed.")

Second searchable corpus package:
/content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/production_smoke/phase2/releases/phase2-smoke-002

Comparing corpus identity
Same corpus: True
Same searchable-window mapping: True
E5 vectors exactly identical: True

PASS: unchanged transcript windows reused their existing E5 embeddings.
E5 model closed.


Imagine we eventually have:

```
~1,478 videos
~30,000 transcript windows
```

and we have already spent time generating their E5 vectors. Later, 10 new videos are added.

We want:
```
old unchanged transcripts → reuse existing embeddings
new/changed transcripts → run E5 only for those
```

not encode ~30,000 documents again.